## 📦 階段 1：安裝與匯入必要套件

In [ ]:
!pip install torch torchvision matplotlib scikit-image tqdm --quiet
import os, random, shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from torchvision.models import vgg16
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.cuda.amp import GradScaler, autocast
from skimage.metrics import structural_similarity as ssim_fn, peak_signal_noise_ratio as psnr_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('✅ 使用裝置:', device)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.2 MB/s eta 0:00:00
✅ 使用裝置: cuda


## 📁 階段 2：資料準備與切分（3000張，85/15）

In [ ]:
# ✅ 下載資料集（僅限第一次執行）
!apt install git-lfs -y
!git clone https://huggingface.co/datasets/MichaelP84/manga-colorization-dataset
%cd manga-colorization-dataset
!git lfs pull
%cd ..

# ✅ 解析 parquet 檔並建立圖片資料夾
import pandas as pd
import io
from PIL import Image
from tqdm import tqdm
import os

os.makedirs("manga_dataset/bw", exist_ok=True)
os.makedirs("manga_dataset/color", exist_ok=True)

# 只取前 3000 張，避免訓練過久
max_images = 3000
count = 0
for file in tqdm(sorted(os.listdir("manga-colorization-dataset/data"))):
    if not file.endswith(".parquet"): continue
    df = pd.read_parquet(f"manga-colorization-dataset/data/{file}")
    for _, row in df.iterrows():
        try:
            bw = Image.open(io.BytesIO(row["bw_image"]["bytes"])).resize((256, 256))
            color = Image.open(io.BytesIO(row["color_image"]["bytes"])).resize((256, 256))
            bw.save(f"manga_dataset/bw/{count:05}.png")
            color.save(f"manga_dataset/color/{count:05}.png")
            count += 1
            if count >= max_images:
                break
        except:
            continue
    if count >= max_images:
        break

print(f"✅ 下載並轉換完成，共 {count} 張")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
Cloning into 'manga-colorization-dataset'...
remote: Enumerating objects: 50, done.
remote: Total 50 (delta 0), reused 0 (delta 0), pack-reused 50 (from 1)
Unpacking objects: 100% (50/50), 8.53 KiB | 1.07 MiB/s, done.
Filtering content: 100% (43/43), 19.71 GiB | 19.37 MiB/s, done.
/content/manga-colorization-dataset
/content


 23%|██▎       | 10/43 [03:46<12:27, 22.65s/it]

✅ 下載並轉換完成，共 3000 張


In [ ]:
os.makedirs("train/bw", exist_ok=True)
os.makedirs("train/color", exist_ok=True)
os.makedirs("val/bw", exist_ok=True)
os.makedirs("val/color", exist_ok=True)

bw_path = "manga_dataset/bw"
color_path = "manga_dataset/color"
files = sorted(os.listdir(bw_path))[:3000]
random.shuffle(files)

split = int(len(files) * 0.85)
train_files = files[:split]
val_files = files[split:]

for i, f in enumerate(tqdm(train_files)):
    shutil.copy(os.path.join(bw_path, f), f"train/bw/{i:05}.png")
    shutil.copy(os.path.join(color_path, f), f"train/color/{i:05}.png")

for i, f in enumerate(tqdm(val_files)):
    shutil.copy(os.path.join(bw_path, f), f"val/bw/{i:05}.png")
    shutil.copy(os.path.join(color_path, f), f"val/color/{i:05}.png")

print(f"✅ 完成切分，訓練集: {len(train_files)}，驗證集: {len(val_files)}")


100%|██████████| 450/450 [00:00<00:00, 4081.37it/s]

✅ 完成切分，訓練集: 2550，驗證集: 450


## 🧱 階段 3：Dataset、模型定義

In [ ]:
class MangaDataset(Dataset):
    def __init__(self, bw_dir, color_dir, augment=True):
        self.bw_images = sorted(os.listdir(bw_dir))
        self.color_images = sorted(os.listdir(color_dir))
        self.bw_dir = bw_dir
        self.color_dir = color_dir
        self.augment = augment

    def __len__(self):
        return len(self.bw_images)

    def __getitem__(self, idx):
        bw_img = Image.open(os.path.join(self.bw_dir, self.bw_images[idx])).convert("L")
        color_img = Image.open(os.path.join(self.color_dir, self.color_images[idx])).convert("RGB")
        if self.augment:
            if random.random() > 0.5:
                bw_img = transforms.functional.hflip(bw_img)
                color_img = transforms.functional.hflip(color_img)
            if random.random() > 0.5:
                bw_img = transforms.functional.vflip(bw_img)
                color_img = transforms.functional.vflip(color_img)

        bw_tensor = transforms.ToTensor()(bw_img)
        color_tensor = transforms.ToTensor()(color_img)
        bw_tensor = transforms.Normalize((0.5,), (0.5,))(bw_tensor)
        color_tensor = transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))(color_tensor)
        return bw_tensor, color_tensor

train_loader = DataLoader(MangaDataset("train/bw", "train/color"), batch_size=16, shuffle=True)
val_loader = DataLoader(MangaDataset("val/bw", "val/color", augment=False), batch_size=8)


In [ ]:
class UNetGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        def down(in_c, out_c, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1)]
            if norm: layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2))
            return nn.Sequential(*layers)

        def up(in_c, out_c):
            return nn.Sequential(
                nn.ConvTranspose2d(in_c, out_c, 4, 2, 1),
                nn.BatchNorm2d(out_c),
                nn.ReLU()
            )

        self.enc1 = down(1, 64, False)
        self.enc2 = down(64, 128)
        self.enc3 = down(128, 256)
        self.enc4 = down(256, 512)
        self.enc5 = down(512, 512)
        self.dec1 = up(512, 512)
        self.dec2 = up(1024, 256)
        self.dec3 = up(512, 128)
        self.dec4 = up(256, 64)
        self.final = nn.Sequential(nn.ConvTranspose2d(128, 3, 4, 2, 1), nn.Tanh())

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)
        d1 = self.dec1(e5)
        d2 = self.dec2(torch.cat([d1, e4], 1))
        d3 = self.dec3(torch.cat([d2, e3], 1))
        d4 = self.dec4(torch.cat([d3, e2], 1))
        return self.final(torch.cat([d4, e1], 1))

class PatchDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        def block(in_c, out_c, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1)]
            if norm: layers.append(nn.InstanceNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2))
            return layers
        self.model = nn.Sequential(
            *block(4, 64, False),
            *block(64, 128),
            *block(128, 256),
            *block(256, 512),
            nn.ZeroPad2d((1,0,1,0)),
            nn.Conv2d(512, 1, 4, padding=1)
        )

    def forward(self, a, b):
        return self.model(torch.cat([a, b], 1))

G = UNetGenerator().to(device)
D = PatchDiscriminator().to(device)


## 🧪 階段 4：三組損失訓練（感知 / 風格 / 感知+風格）

In [ ]:
# ✅ 匯入必要模組
import os, torch, random
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchvision.utils import save_image
from torch.cuda.amp import GradScaler, autocast
from skimage.metrics import structural_similarity as ssim_fn, peak_signal_noise_ratio as psnr_fn

# ✅ 感知 / 風格損失（使用 VGG）
from torchvision.models import vgg16
vgg = vgg16(weights='IMAGENET1K_V1').features[:16].eval().cuda()
for p in vgg.parameters(): p.requires_grad = False

def perceptual_loss(fake, real):
    return nn.L1Loss()(vgg(fake), vgg(real))

def gram_matrix(f):
    b, c, h, w = f.size()
    f = f.view(b, c, -1)
    return torch.bmm(f, f.transpose(1, 2)) / (c * h * w)

def style_loss(fake, real):
    G_fake = gram_matrix(vgg(fake))
    G_real = gram_matrix(vgg(real))
    return nn.L1Loss()(G_fake, G_real)

# ✅ 假設 G, D, train_loader, val_loader 已建立好
# G: UNetGenerator, D: PatchDiscriminator
# 訓練參數
epochs = 200
save_interval = 10
max_train = 3000
scaler = GradScaler()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion_GAN = nn.MSELoss()
criterion_L1 = nn.L1Loss()

# ✅ 三組訓練配置
configs = [
    {"name": "PerceptualOnly", "lambda_L1": 50, "lambda_per": 10, "lambda_style": 0},
    {"name": "StyleOnly", "lambda_L1": 50, "lambda_per": 0, "lambda_style": 100},
    {"name": "PerceptualStyle", "lambda_L1": 50, "lambda_per": 5, "lambda_style": 100},
]

for config in configs:
    name = config["name"]
    λ1 = config["lambda_L1"]
    λp = config["lambda_per"]
    λs = config["lambda_style"]

    G.apply(lambda m: m.reset_parameters() if hasattr(m, 'reset_parameters') else None)
    D.apply(lambda m: m.reset_parameters() if hasattr(m, 'reset_parameters') else None)

    opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

    gen_losses, disc_losses, l1_losses = [], [], []
    ssim_scores, psnr_scores = [], []

    os.makedirs(f"samples/{name}", exist_ok=True)
    os.makedirs(f"checkpoints/{name}", exist_ok=True)

    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        total_g, total_d, total_l1 = 0, 0, 0

        for i, (bw_imgs, color_imgs) in enumerate(tqdm(train_loader, desc=f"[{name}] Epoch {epoch}/{epochs}")):
            if i * bw_imgs.size(0) > max_train: break
            bw_imgs, color_imgs = bw_imgs.to(device), color_imgs.to(device)

            with autocast():
                fake = G(bw_imgs)
                real_pred = D(bw_imgs, color_imgs)
                fake_pred = D(bw_imgs, fake.detach())
                d_loss = (criterion_GAN(real_pred, torch.ones_like(real_pred)) +
                          criterion_GAN(fake_pred, torch.zeros_like(fake_pred))) * 0.5

            opt_D.zero_grad()
            scaler.scale(d_loss).backward()
            scaler.step(opt_D)

            with autocast():
                fake = G(bw_imgs)
                pred_fake = D(bw_imgs, fake)
                gan_loss = criterion_GAN(pred_fake, torch.ones_like(pred_fake))
                l1 = criterion_L1(fake, color_imgs)
                p = perceptual_loss(fake, color_imgs) if λp > 0 else 0
                s = style_loss(fake, color_imgs) if λs > 0 else 0
                g_loss = gan_loss + λ1 * l1 + λp * p + λs * s

            opt_G.zero_grad()
            scaler.scale(g_loss).backward()
            scaler.step(opt_G)
            scaler.update()

            total_g += g_loss.item()
            total_d += d_loss.item()
            total_l1 += l1.item()

        avg_g = total_g / len(train_loader)
        avg_d = total_d / len(train_loader)
        avg_l1 = total_l1 / len(train_loader)
        gen_losses.append(avg_g)
        disc_losses.append(avg_d)
        l1_losses.append(avg_l1)

        # === 驗證 SSIM / PSNR ===
        G.eval()
        ssim_sum = psnr_sum = count = 0
        with torch.no_grad():
            for bw_imgs, color_imgs in val_loader:
                bw_imgs, color_imgs = bw_imgs.to(device), color_imgs.to(device)
                fake = G(bw_imgs)
                for i in range(len(fake)):
                  f = fake[i].cpu().permute(1, 2, 0).numpy()
                  t = color_imgs[i].cpu().permute(1, 2, 0).numpy()
                  f = (f * 0.5 + 0.5).clip(0, 1)
                  t = (t * 0.5 + 0.5).clip(0, 1)
                  ssim_sum += ssim_fn(f, t, channel_axis=-1, data_range=1.0)
                  psnr_sum += psnr_fn(t, f, data_range=1.0)
                  count += 1

        avg_ssim = ssim_sum / count
        avg_psnr = psnr_sum / count
        ssim_scores.append(avg_ssim)
        psnr_scores.append(avg_psnr)

        print(f"[{name}] Epoch {epoch:03} | G_Loss={avg_g:.4f} | D_Loss={avg_d:.4f} | SSIM={avg_ssim:.4f} | PSNR={avg_psnr:.2f} dB")

        if epoch % save_interval == 0:
            val_sample = next(iter(val_loader))
            bw_val, color_val = val_sample[0][:20].to(device), val_sample[1][:20].to(device)
            with torch.no_grad():
                fake_val = G(bw_val)
            save_image((bw_val + 1) / 2, f"samples/{name}/epoch{epoch:03}_bw.png", nrow=5)
            save_image((fake_val + 1) / 2, f"samples/{name}/epoch{epoch:03}_pred.png", nrow=5)
            save_image((color_val + 1) / 2, f"samples/{name}/epoch{epoch:03}_gt.png", nrow=5)
            torch.save(G.state_dict(), f"checkpoints/{name}/G_epoch{epoch}.pth")

    # === 繪圖 ===
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(gen_losses, label="G_Loss")
    plt.plot(disc_losses, label="D_Loss")
    plt.plot(l1_losses, label="L1_Loss")
    plt.title(f"{name} - Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(ssim_scores, label="SSIM")
    plt.plot(psnr_scores, label="PSNR")
    plt.title(f"{name} - SSIM/PSNR")
    plt.xlabel("Epoch"); plt.ylabel("Score"); plt.legend()
    plt.tight_layout()
    plt.savefig(f"samples/{name}/metrics.png")
    plt.close()

<ipython-input-9-23088e9b42fa>:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
[PerceptualOnly] Epoch 1/200:   0%|          | 0/160 [00:00<?, ?it/s]<ipython-input-9-23088e9b42fa>:72: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
<ipython-input-9-23088e9b42fa>:83: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
[PerceptualOnly] Epoch 1/200: 100%|██████████| 160/160 [00:50<00:00,  3.18it/s]


[PerceptualOnly] Epoch 001 | G_Loss=18.8670 | D_Loss=0.2495 | SSIM=0.8398 | PSNR=20.71 dB


[PerceptualOnly] Epoch 2/200: 100%|██████████| 160/160 [00:49<00:00,  3.20it/s]


[PerceptualOnly] Epoch 002 | G_Loss=12.2405 | D_Loss=0.1462 | SSIM=0.8887 | PSNR=21.44 dB


[PerceptualOnly] Epoch 3/200: 100%|██████████| 160/160 [00:49<00:00,  3.21it/s]


...

[PerceptualStyle] Epoch 123/200: 100%|██████████| 160/160 [01:08<00:00,  2.35it/s]


[PerceptualStyle] Epoch 123 | G_Loss=nan | D_Loss=0.0092 | SSIM=0.9510 | PSNR=23.48 dB


[PerceptualStyle] Epoch 124/200: 100%|██████████| 160/160 [01:08<00:00,  2.34it/s]


[PerceptualStyle] Epoch 124 | G_Loss=nan | D_Loss=0.1334 | SSIM=0.9512 | PSNR=23.53 dB


[PerceptualStyle] Epoch 125/200:  58%|█████▊    | 93/160 [00:39<00:29,  2.29it/s]

In [ ]:
# ✅ 匯入必要模組
import os, torch, random
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchvision.utils import save_image
from torch.cuda.amp import GradScaler, autocast
from skimage.metrics import structural_similarity as ssim_fn, peak_signal_noise_ratio as psnr_fn

# ✅ 感知 / 風格損失（使用 VGG）
from torchvision.models import vgg16
vgg = vgg16(weights='IMAGENET1K_V1').features[:16].eval().cuda()
for p in vgg.parameters(): p.requires_grad = False

def perceptual_loss(fake, real):
    return nn.L1Loss()(vgg(fake), vgg(real))

def gram_matrix(f):
    b, c, h, w = f.size()
    f = f.view(b, c, -1)
    return torch.bmm(f, f.transpose(1, 2)) / (c * h * w)

def style_loss(fake, real):
    G_fake = gram_matrix(vgg(fake))
    G_real = gram_matrix(vgg(real))
    return nn.L1Loss()(G_fake, G_real)

# ✅ 假設 G, D, train_loader, val_loader 已建立好
# G: UNetGenerator, D: PatchDiscriminator
# 訓練參數
epochs = 200
save_interval = 10
max_train = 3000
scaler = GradScaler()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion_GAN = nn.MSELoss()
criterion_L1 = nn.L1Loss()

# ✅ 設定訓練配置（僅保留 StyleOnly / PerceptualStyle）
configs = [
    # {"name": "PerceptualOnly", "lambda_L1": 50, "lambda_per": 10, "lambda_style": 0},  # ❌ 暫時關閉
    {"name": "StyleOnly", "lambda_L1": 50, "lambda_per": 0, "lambda_style": 100},        # ✅ 保留
    {"name": "PerceptualStyle", "lambda_L1": 50, "lambda_per": 5, "lambda_style": 100},  # ✅ 保留
]

for config in configs:
    name = config["name"]
    λ1 = config["lambda_L1"]
    λp = config["lambda_per"]
    λs = config["lambda_style"]

    G.apply(lambda m: m.reset_parameters() if hasattr(m, 'reset_parameters') else None)
    D.apply(lambda m: m.reset_parameters() if hasattr(m, 'reset_parameters') else None)

    opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

    gen_losses, disc_losses, l1_losses = [], [], []
    ssim_scores, psnr_scores = [], []

    os.makedirs(f"samples/{name}", exist_ok=True)
    os.makedirs(f"checkpoints/{name}", exist_ok=True)

    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        total_g, total_d, total_l1 = 0, 0, 0

        for i, (bw_imgs, color_imgs) in enumerate(tqdm(train_loader, desc=f"[{name}] Epoch {epoch}/{epochs}")):
            if i * bw_imgs.size(0) > max_train: break
            bw_imgs, color_imgs = bw_imgs.to(device), color_imgs.to(device)

            with autocast():
                fake = G(bw_imgs)
                real_pred = D(bw_imgs, color_imgs)
                fake_pred = D(bw_imgs, fake.detach())
                d_loss = (criterion_GAN(real_pred, torch.ones_like(real_pred)) +
                          criterion_GAN(fake_pred, torch.zeros_like(fake_pred))) * 0.5

            opt_D.zero_grad()
            scaler.scale(d_loss).backward()
            scaler.step(opt_D)

            with autocast():
                fake = G(bw_imgs)
                pred_fake = D(bw_imgs, fake)
                gan_loss = criterion_GAN(pred_fake, torch.ones_like(pred_fake))
                l1 = criterion_L1(fake, color_imgs)
                p = perceptual_loss(fake, color_imgs) if λp > 0 else 0
                s = style_loss(fake, color_imgs) if λs > 0 else 0
                g_loss = gan_loss + λ1 * l1 + λp * p + λs * s

            opt_G.zero_grad()
            scaler.scale(g_loss).backward()
            scaler.step(opt_G)
            scaler.update()

            total_g += g_loss.item()
            total_d += d_loss.item()
            total_l1 += l1.item()

        avg_g = total_g / len(train_loader)
        avg_d = total_d / len(train_loader)
        avg_l1 = total_l1 / len(train_loader)
        gen_losses.append(avg_g)
        disc_losses.append(avg_d)
        l1_losses.append(avg_l1)

        # === 驗證 SSIM / PSNR ===
        G.eval()
        ssim_sum = psnr_sum = count = 0
        with torch.no_grad():
            for bw_imgs, color_imgs in val_loader:
                bw_imgs, color_imgs = bw_imgs.to(device), color_imgs.to(device)
                fake = G(bw_imgs)
                for i in range(len(fake)):
                    f = fake[i].cpu().permute(1, 2, 0).numpy()
                    t = color_imgs[i].cpu().permute(1, 2, 0).numpy()
                    f = (f * 0.5 + 0.5).clip(0, 1)
                    t = (t * 0.5 + 0.5).clip(0, 1)
                    ssim_sum += ssim_fn(f, t, channel_axis=-1, data_range=1.0)
                    psnr_sum += psnr_fn(t, f, data_range=1.0)
                    count += 1

        avg_ssim = ssim_sum / count
        avg_psnr = psnr_sum / count
        ssim_scores.append(avg_ssim)
        psnr_scores.append(avg_psnr)

        print(f"[{name}] Epoch {epoch:03} | G_Loss={avg_g:.4f} | D_Loss={avg_d:.4f} | SSIM={avg_ssim:.4f} | PSNR={avg_psnr:.2f} dB")

        if epoch % save_interval == 0:
            val_sample = next(iter(val_loader))
            bw_val, color_val = val_sample[0][:20].to(device), val_sample[1][:20].to(device)
            with torch.no_grad():
                fake_val = G(bw_val)
            save_image((bw_val + 1) / 2, f"samples/{name}/epoch{epoch:03}_bw.png", nrow=5)
            save_image((fake_val + 1) / 2, f"samples/{name}/epoch{epoch:03}_pred.png", nrow=5)
            save_image((color_val + 1) / 2, f"samples/{name}/epoch{epoch:03}_gt.png", nrow=5)
            torch.save(G.state_dict(), f"checkpoints/{name}/G_epoch{epoch}.pth")

    # === 繪圖 ===
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(gen_losses, label="G_Loss")
    plt.plot(disc_losses, label="D_Loss")
    plt.plot(l1_losses, label="L1_Loss")
    plt.title(f"{name} - Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(ssim_scores, label="SSIM")
    plt.plot(psnr_scores, label="PSNR")
    plt.title(f"{name} - SSIM/PSNR")
    plt.xlabel("Epoch"); plt.ylabel("Score"); plt.legend()
    plt.tight_layout()
    plt.savefig(f"samples/{name}/metrics.png")
    plt.close()
